In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import time
import joblib
import pandas as pd

sys.path.append('../src')

# Валидация GigaChat с помощью библиотеки llm_val

llm_val - библиотека, разрабатываемая УМР для валидации применений GigaChat.

## Основные концепции

## Пример валидации

In [3]:
import joblib

In [4]:
data = joblib.load('data/data_mmlu.pkl')

In [5]:
data.head()

,query,answer,true_answer
0,\nКонтекст:\nЧто такое ипотека и ипотечный кре...,A) Жилое помещение,['D']
1,\nКонтекст:\nКредитные карты.\n\nКакие действи...,D) 60 месяцев,['A']
2,\nКонтекст:\nПриобретение готового жилья\nГото...,C) Отсутствие возможности оформления постоянно...,['C']
3,\nКонтекст:\nПриобретение строящегося жилья.\n...,[C] 30%,['C']
4,\nКонтекст:\nКоммерческая недвижимость\n\nУсло...,[A] Возраст заемщика от 21 до 75 лет.,['A']


Для проведения валидации необходимо два обязательных объекта: Sampler и Scorer.

### Написание sampler'a

Начнем с sampler'а. Допустим, что под наши данные не подошли уже существующие sampler'ы, тогда напишем **свой sampler**. *Подробная документация по написанию sampler'а лежит в `examples/3.1.Write_Sampler_example.ipynb`*. 

In [6]:
import re
import pandas as pd
from llm_val.sampler.Sampler import Sampler

class MySampler(Sampler):
    def __init__(self, test):
        test = self._preproc(test)
        super().__init__(test=test)

    def _preproc(self, var):
        X = pd.DataFrame()
        y = pd.DataFrame()
        X['question'] = var['query'].apply(self._parse_question)
        X['context'] = var['query'].apply(self._parse_context)
        X['answer'] = var['query'].apply(self._parse_answer)
        X['true_answer'] = var['true_answer'].apply(self._parse_true_answer)
        X['model_answer'] = var['answer'].apply(self._parse_model_answer)
        y['is_correct'] = (X['true_answer'] == X['model_answer']).astype(int)
        return {'X': X, 'y': y}

    def _parse_question(self, text):
        pattern = r'Вопрос:\s*(.+?)(?=\nВарианты ответов:)'
        question_match = re.search(pattern, text, re.DOTALL)
        question = question_match.group(1) if question_match else None
        return question

    def _parse_true_answer(self, true_answ):
        replace_dict = {"['A']": 'A',
                        "['B']": 'B',
                        "['C']": 'C',
                        "['D']": 'D'}
        return replace_dict[true_answ]

    def _parse_model_answer(self, text):
        pattern = r'^\[?([A-Z])\]?'
        model_match = re.search(pattern, text, re.MULTILINE)
        model_answ = model_match.group(1) if model_match else None
        return model_answ

    def _parse_context(self, text):
        pattern = r'Контекст:?\n(.*?)Вопрос:'
        context_match = re.search(pattern, text, re.DOTALL)
        context = context_match.group(1) if context_match else None
        context_list = [i for i in context.split('\n') if len(i.strip()) != 0]
        return context_list

    def _parse_answer(self, text):
        pattern = r'Варианты ответов:\s*(.+?)(?=\nТвой ответ:)'
        answer_match = re.search(pattern, text, re.DOTALL)
        answer = answer_match.group(1) if answer_match else None
        answer_list = [i for i in answer.split('\n') if len(i.strip()) != 0]
        answer_dict = {self._parse_model_answer(i): self._parse_answer_text(i) for i in answer_list}
        return answer_dict

    def _parse_answer_text(self, text):
        pattern = r'^\[?[A-Z][\)\]]?\s*(.*)'
        answer_match = re.search(pattern, text)
        answer = answer_match.group(1) if answer_match else None
        return answer

In [7]:
sampler = MySampler(data)

### Написание scorer'а

Перейдем к scorer'у. Допустим, что нам подходит уже готовый `UniversalScorer` и готовые метрики из `METRICS`. *Подробная документация к scorer'у лежит в файле `examples/4.1.Write_Scorer_example.ipynb`.*

In [8]:
from llm_val.utils.metrics import METRICS
from llm_val.scorer.UniversalScorer import UniversalScorer

scorer = UniversalScorer(metrics=METRICS)

### Запуск тестов

После создания sampler'а и scorer'а можно запускать тесты. *Подробная документация по использованию тестов лежит в `examples/5.0.Test_example.ipynb`.*

In [9]:
from llm_val.val_test.model_quality.valtest_metric import valtest_metric
from llm_val.val_test.model_quality.valtest_stat_significance import valtest_stat_significance

In [12]:
res_valtest_metric = valtest_metric(sampler=sampler,
                                    scorer=scorer,
                                    data_type='test',
                                    metric_name='single_mean',
                                    semaphore_threshold={'default': (0.2, 0.4)},
                                    greater_is_better={'default': True},
                                    is_info=False,
                                    metric_value=None)

In [13]:
res_valtest_stat_significance = valtest_stat_significance(sampler=sampler,
                                                          data_type='test',
                                                          main_column=None,
                                                          value_threshold=0.5,
                                                          extra_samplers=None,
                                                          same_questions=False,
                                                          semaphore_threshold=0.01,
                                                          is_info=False,
                                                          p_value=None)

Посмотрим результаты и выведем табличку по каждому тесту.

In [14]:
res_valtest_metric['report']['result_dataframes'][0]

,Значение метрики,Результат теста
single_mean_is_correct,0.6,green


In [15]:
res_valtest_stat_significance['report']['result_dataframes'][0]

,Значение p-value на test,Результат теста
0,0.704,red


Вот и всё, теперь вы умеете валидировать GigaChat😎

Подробную документацию по каждой сущности(абстракции) можно посмотреть в ноутбуках в директории `examples`.